# V9: high-SNR fine-tuning from V8 (P=10)

Keeps the V8 40-layer model and data fixed. Only 25 and 30 dB training examples receive 4x per-sample NMSE weight. Adam starts at 1e-5. The selected checkpoint minimizes the worst validation DL-minus-GS gap over all eight SNRs. Test data is evaluated only after selection. This experiment may improve high-SNR results; it cannot guarantee beating GS.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
import os, torch
root = '/content/gdrive/MyDrive'
required = [f'{root}/DL_dataset_RSR25dB/{split}_P10.mat' for split in ('train','validation','test')]
required += [f'{root}/DL_dataset_RSR25dB/v6_starting_models/gs_{split}_P10.json' for split in ('validation','test')]
required += [f'{root}/unfolded_dl_v8_continue_p10_results/model_P10_seed{seed}_v8.pt' for seed in (20261026,20261027,20261028)]
missing = [p for p in required if not os.path.isfile(p)]
assert not missing, f'Missing Drive files: {missing}'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Connect a T4 GPU before training')
print('P=10 data, GS curves, and all V8 checkpoints found.')

## Upload the six scripts
Upload these from the `v2` folder: `colab_unfolded_dl_v3_highsnr.py`, `colab_unfolded_dl_v5_snr_aware.py`, `colab_unfolded_dl_v6_gs_gap.py`, `colab_unfolded_dl_v7_continue_p10.py`, `colab_unfolded_dl_v8_continue_p10.py`, and `colab_unfolded_dl_v9_highsnr_finetune.py`.

In [ ]:
from google.colab import files
uploaded = files.upload()
required_scripts = {
 'colab_unfolded_dl_v3_highsnr.py',
 'colab_unfolded_dl_v5_snr_aware.py',
 'colab_unfolded_dl_v6_gs_gap.py',
 'colab_unfolded_dl_v7_continue_p10.py',
 'colab_unfolded_dl_v8_continue_p10.py',
 'colab_unfolded_dl_v9_highsnr_finetune.py',
}
missing = required_scripts.difference(uploaded)
assert not missing, f'Upload these scripts: {sorted(missing)}'
print('All six scripts uploaded.')

## Train
In Colab choose **Runtime → Change runtime type → T4 GPU**, connect, then run this cell. The run saves a resumable checkpoint after every epoch to Drive.

In [ ]:
assert torch.cuda.is_available(), 'Connect a T4 GPU runtime first.'
!python /content/colab_unfolded_dl_v9_highsnr_finetune.py

## Summarize and compare the independent test results
Negative DL-GS means DL has lower NMSE. The V8 and V9 metrics below use the same saved test set and seeds.

In [ ]:
import json, os
import numpy as np
root = '/content/gdrive/MyDrive'
v8_dir = f'{root}/unfolded_dl_v8_continue_p10_results'
v9_dir = f'{root}/unfolded_dl_v9_highsnr_finetune_p10_results'
v8 = [torch.load(f'{v8_dir}/model_P10_seed{s}_v8.pt', map_location='cpu', weights_only=False)['result'] for s in (20261026,20261027,20261028)]
v9 = json.load(open(f'{v9_dir}/summary_unfolded_v9_highsnr_p10.json', encoding='utf-8'))
snrs = [-5,0,5,10,15,20,25,30]
print('SNR | V8 mean DL-GS | V9 mean DL-GS | V9 seeds beating GS')
for snr in snrs:
 k = str(snr)
 g8 = [r['test_dl_minus_gs_db_by_snr'][k] for r in v8]
 g9 = [r['test_dl_minus_gs_db_by_snr'][k] for r in v9]
 print(f'{snr:>3} dB | {np.mean(g8):+8.3f} dB | {np.mean(g9):+8.3f} dB | {sum(x < 0 for x in g9)}/3')
print('V9 results:', v9_dir)